# X-Ray Disease Dectector with DenseNet-121 and Grad-CAM

## Imports

In [ ]:
import os
import gc
import time
import random
import itertools
import warnings
import json
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import densenet121, DenseNet121_Weights

from datasets import load_dataset
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

warnings.filterwarnings("ignore")
print("PyTorch:", torch.__version__)


## Configuration

In [ ]:
SEED = 42
DATASET_NAME = "arudaev/chest-xray-14-320"
TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 250, 50, 50  
IMAGE_SIZE = 160
BATCH_SIZE = 32
NUM_WORKERS = 0  # safest default for notebooks/Windows; increase after validating locally
COMPARISON_EPOCHS = 10  # validation-driven early stopping may end a run sooner
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_REDUCTION_FACTOR = 0.5
LR_REDUCTION_PATIENCE = 1  # reduce learning rate after this many non-improving validation epochs
EARLY_STOPPING_PATIENCE = 4  # stop only after the scheduler has time to improve the run
MIN_LEARNING_RATE = 1e-6
MAX_POS_WEIGHT = 20.0
FOCAL_ALPHA, FOCAL_GAMMA = 0.25, 2.0
QUALITY_TOLERANCE = 0.005  # prefer the faster run when validation auc-roc values are within this margin
SELECTED_LOSS = "weighted_bce"  # selected in the earlier loss-screening experiment
# Use "quick" during development; switch to "thorough" only for final reported timing.
BENCHMARK_PRESET = "quick"
BENCHMARK_SETTINGS = {
    "quick": {"warmup_batches": 2, "timing_batches": 4, "gradcam_warmup": 1, "gradcam_samples": 3},
    "thorough": {"warmup_batches": 5, "timing_batches": 20, "gradcam_warmup": 5, "gradcam_samples": 10},
}
if BENCHMARK_PRESET not in BENCHMARK_SETTINGS:
    raise ValueError("BENCHMARK_PRESET must be 'quick' or 'thorough'")
BENCHMARK = BENCHMARK_SETTINGS[BENCHMARK_PRESET]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
USE_CHANNELS_LAST = DEVICE.type == "cuda"  # measured in the benchmark; disable if it loses on your GPU
PIN_MEMORY = DEVICE.type == "cuda"
PROJECT_ROOT = Path.cwd()
# Supports launching Jupyter from either the repository root or notebooks/.
if not (PROJECT_ROOT / "src").is_dir() and (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
RUN_ID = f"densenet121_full_vs_partial_{TRAIN_SIZE/1000:.2f}k_seed{SEED}"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "runs" / RUN_ID
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
METRICS_DIR = ARTIFACT_DIR / "metrics"
MANIFEST_DIR = ARTIFACT_DIR / "manifests"
BENCHMARK_DIR = ARTIFACT_DIR / "benchmarks"
FIGURE_DIR = ARTIFACT_DIR / "figures"
CONFIG_DIR = ARTIFACT_DIR / "config"
REPORT_DIR = ARTIFACT_DIR / "reports"
for directory in (CHECKPOINT_DIR, METRICS_DIR, MANIFEST_DIR, BENCHMARK_DIR, FIGURE_DIR, CONFIG_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

run_config = {
    "seed": SEED, "dataset": DATASET_NAME, "train_size": TRAIN_SIZE, "validation_size": VAL_SIZE,
    "test_size": TEST_SIZE, "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE,
    "max_epochs": COMPARISON_EPOCHS, "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
    "lr_reduction_factor": LR_REDUCTION_FACTOR, "lr_reduction_patience": LR_REDUCTION_PATIENCE,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE, "min_learning_rate": MIN_LEARNING_RATE,
    "loss": SELECTED_LOSS, "quality_tolerance": QUALITY_TOLERANCE, "device": str(DEVICE),
    "use_amp": USE_AMP, "use_channels_last": USE_CHANNELS_LAST,
    "benchmark_preset": BENCHMARK_PRESET, "benchmark_settings": BENCHMARK,
}
(CONFIG_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True  # fixed image size; favors realistic throughput timing

print({"device": str(DEVICE), "amp": USE_AMP, "channels_last": USE_CHANNELS_LAST,
       "train/val/test": (TRAIN_SIZE, VAL_SIZE, TEST_SIZE), "image_size": IMAGE_SIZE})


## Labels, reproducible subset loading and label encoding

In [ ]:
DISEASES = ["Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass", "Nodule",
            "Pneumonia", "Pneumothorax", "Consolidation", "Edema", "Emphysema", "Fibrosis",
            "Pleural_Thickening", "Hernia"]
NUM_CLASSES = len(DISEASES)
DISEASE_TO_INDEX = {name: i for i, name in enumerate(DISEASES)}

def encode_labels(labels):
    if isinstance(labels, str):
        labels = labels.split("|")
    target = np.zeros(NUM_CLASSES, dtype=np.float32)
    for label in labels or []:
        label = str(label).strip()
        if label in DISEASE_TO_INDEX:
            target[DISEASE_TO_INDEX[label]] = 1.0
    return target

def take_samples(split, n):
    stream = load_dataset(DATASET_NAME, split=split, streaming=True)
    samples = list(itertools.islice(stream, n))
    if len(samples) != n:
        raise RuntimeError(f"Requested {n} {split} samples but received {len(samples)}")
    return samples

train_samples = take_samples("train", TRAIN_SIZE)
val_samples = take_samples("validation", VAL_SIZE)
test_samples = take_samples("test", TEST_SIZE)  # never inspected for model selection
def sample_id(sample, split, position):
    # Prefer an upstream identifier when available, but retain a stable fallback.
    for key in ("id", "image_id", "imageId", "Image Index", "path", "filename"):
        if key in sample and sample[key] is not None:
            return str(sample[key])
    return f"{split}_{position:06d}"

def targets_for(samples):
    return np.stack([encode_labels(sample["labels"]) for sample in samples])

train_targets_np, val_targets_np, test_targets_np = targets_for(train_samples), targets_for(val_samples), targets_for(test_samples)
positive_counts = train_targets_np.sum(axis=0)
pos_weights_np = np.clip((TRAIN_SIZE - positive_counts) / np.maximum(positive_counts, 1), 1.0, MAX_POS_WEIGHT)
pos_weights = torch.tensor(pos_weights_np, dtype=torch.float32, device=DEVICE)

manifest = pd.concat([
    pd.DataFrame({"split": split, "position": np.arange(len(samples)), "sample_id": [sample_id(s, split, i) for i, s in enumerate(samples)]})
    for split, samples in (("train", train_samples), ("validation", val_samples), ("test", test_samples))
], ignore_index=True)
manifest.insert(0, "seed", SEED)
manifest.to_csv(MANIFEST_DIR / "split_manifest.csv", index=False)

prevalence_rows = []
for split, targets in (("train", train_targets_np), ("validation", val_targets_np), ("test", test_targets_np)):
    for index, disease in enumerate(DISEASES):
        prevalence_rows.append({"split": split, "disease": disease, "positive_count": int(targets[:, index].sum()),
                                "sample_count": len(targets), "prevalence": float(targets[:, index].mean())})
class_prevalence = pd.DataFrame(prevalence_rows)
class_prevalence.to_csv(METRICS_DIR / "class_prevalence.csv", index=False)
display(class_prevalence.pivot(index="disease", columns="split", values="prevalence").round(4))
display(pd.DataFrame({"disease": DISEASES, "positive_train": positive_counts.astype(int), "pos_weight": pos_weights_np.round(2)}))


## Dataset and loaders

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class ChestXrayDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples, self.transform = samples, transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, index):
        sample = self.samples[index]
        image = sample["image"]
        if not isinstance(image, Image.Image):
            image = Image.fromarray(np.asarray(image))
        image = self.transform(image.convert("RGB"))
        return image, torch.tensor(encode_labels(sample["labels"]), dtype=torch.float32), index

train_dataset = ChestXrayDataset(train_samples, train_transform)
val_dataset = ChestXrayDataset(val_samples, eval_transform)
test_dataset = ChestXrayDataset(test_samples, eval_transform)

def make_loader(dataset, shuffle=False):
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=PIN_MEMORY, persistent_workers=NUM_WORKERS > 0)

train_loader, val_loader, test_loader = make_loader(train_dataset, True), make_loader(val_dataset), make_loader(test_dataset)


## Model, losses, metrics and training helpers

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
        super().__init__(); self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p_t = torch.exp(-bce)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - p_t).pow(self.gamma) * bce).mean()

def make_criterion(loss_name):
    if loss_name == "bce": return nn.BCEWithLogitsLoss()
    if loss_name == "weighted_bce": return nn.BCEWithLogitsLoss(pos_weight=pos_weights)
    if loss_name == "focal": return FocalLoss()
    raise ValueError(f"Unknown loss: {loss_name}")

def set_trainable(model, mode):
    for p in model.parameters(): p.requires_grad = False
    if mode == "full":
        for p in model.parameters(): p.requires_grad = True
    elif mode == "partial":
        for module in (model.features.denseblock4, model.features.norm5, model.classifier):
            for p in module.parameters(): p.requires_grad = True
    elif mode == "frozen":
        for p in model.classifier.parameters(): p.requires_grad = True
    else: raise ValueError("mode must be full, partial, or frozen")

def build_model(mode, optimized):
    model = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
    model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)
    set_trainable(model, mode)
    model = model.to(DEVICE)
    if optimized and USE_CHANNELS_LAST:
        model = model.to(memory_format=torch.channels_last)
    return model

def valid_auc(y_true, y_prob, metric):
    values = []
    for c in range(NUM_CLASSES):
        if np.unique(y_true[:, c]).size == 2:
            values.append(roc_auc_score(y_true[:, c], y_prob[:, c]) if metric == "auc" else average_precision_score(y_true[:, c], y_prob[:, c]))
    return float(np.mean(values)) if values else float("nan")

@torch.inference_mode()
def predict(model, loader):
    model.eval(); probs, targets, indices = [], [], []
    for images, labels, idx in loader:
        images = images.to(DEVICE, non_blocking=PIN_MEMORY)
        if USE_CHANNELS_LAST and next(model.parameters()).is_contiguous(memory_format=torch.channels_last): images = images.contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP): logits = model(images)
        probs.append(torch.sigmoid(logits).float().cpu().numpy()); targets.append(labels.numpy()); indices.append(idx.numpy())
    return np.concatenate(targets), np.concatenate(probs), np.concatenate(indices)

def train_experiment(name, mode, loss_name, optimized, epochs):
    torch.manual_seed(SEED)  # same initialization and data-order seed for every run
    if DEVICE.type == "cuda": torch.cuda.manual_seed_all(SEED); torch.cuda.reset_peak_memory_stats()
    model = build_model(mode, optimized)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion, scaler = make_criterion(loss_name), torch.amp.GradScaler("cuda", enabled=USE_AMP)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=LR_REDUCTION_FACTOR, patience=LR_REDUCTION_PATIENCE,
        min_lr=MIN_LEARNING_RATE, threshold=1e-4,
    )
    best_state, best_auc, best_epoch, epochs_without_improvement, history = None, -np.inf, 0, 0, []
    start_total = time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train(); running_loss, images_seen = 0.0, 0; start_epoch = time.perf_counter()
        for images, labels, _ in train_loader:
            images, labels = images.to(DEVICE, non_blocking=PIN_MEMORY), labels.to(DEVICE, non_blocking=PIN_MEMORY)
            if optimized and USE_CHANNELS_LAST: images = images.contiguous(memory_format=torch.channels_last)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP): loss = criterion(model(images), labels)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            running_loss += loss.item() * len(labels); images_seen += len(labels)
        y_val, p_val, _ = predict(model, val_loader)
        auc, ap = valid_auc(y_val, p_val, "auc"), valid_auc(y_val, p_val, "ap")
        epoch_seconds = time.perf_counter() - start_epoch
        scheduler.step(auc)
        history.append({"epoch": epoch, "train_loss": running_loss / images_seen, "val_macro_auc_roc": auc,
                        "val_macro_auprc": ap, "epoch_seconds": epoch_seconds,
                        "learning_rate": optimizer.param_groups[0]["lr"]})
        if auc > best_auc + 1e-4:
            best_auc, best_state, best_epoch, epochs_without_improvement = auc, deepcopy(model.state_dict()), epoch, 0
        else:
            epochs_without_improvement += 1
        print(f"{name} | epoch {epoch}/{epochs} | loss {history[-1]['train_loss']:.4f} | val auc-roc {auc:.4f} | lr {history[-1]['learning_rate']:.2e} | {epoch_seconds:.1f}s")
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"{name} | early stopping after {epoch} epochs; best validation auc-roc was at epoch {best_epoch}.")
            break
    total_seconds = time.perf_counter() - start_total
    model.load_state_dict(best_state)
    final_y_val, final_p_val, _ = predict(model, val_loader)
    checkpoint = CHECKPOINT_DIR / f"{name}.pt"; torch.save(model.state_dict(), checkpoint)
    history_path = METRICS_DIR / f"{name}_history.csv"
    pd.DataFrame(history).to_csv(history_path, index=False)
    return {"name": name, "mode": mode, "loss": loss_name, "optimized": optimized, "model": model, "checkpoint": checkpoint,
            "val_targets": final_y_val, "val_probs": final_p_val, "best_val_auc_roc": best_auc,
            "best_epoch": best_epoch, "epochs_completed": len(history), "history_path": history_path,
            "total_train_seconds": total_seconds, "mean_epoch_seconds": float(np.mean([h['epoch_seconds'] for h in history])),
            "peak_memory_gb": (torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else np.nan), "history": history,
            "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad), "total_parameters": sum(p.numel() for p in model.parameters())}


## Protocol

In [ ]:
# Loss type was selected in earlier loss-screening experiments and fixed for final experiments. 
print("Fixed loss:", SELECTED_LOSS)
print("Saved exact sample IDs:", MANIFEST_DIR / "split_manifest.csv")
print("Saved class prevalence:", METRICS_DIR / "class_prevalence.csv")


## Main speed-quality comparison

In [ ]:
# The three configurations isolate two effects:
# full FP32 -> partial FP32: effect of partial fine-tuning alone
# partial FP32 -> partial optimized: effect of AMP + channels_last
# All use the same loss, split, seed and epoch budget.
experiments = [
    ("full_ft_fp32", "full", False),
    ("partial_ft_fp32", "partial", False),
    ("partial_ft_optm", "partial", True),
]
comparison = [train_experiment(name, mode, SELECTED_LOSS, optimized, COMPARISON_EPOCHS) for name, mode, optimized in experiments]
full_reference = next(r for r in comparison if r["name"] == "full_ft_fp32")
target_auc_roc = full_reference["best_val_auc_roc"] - QUALITY_TOLERANCE

def time_to_target(history, target):
    elapsed = 0.0
    for row in history:
        elapsed += row["epoch_seconds"]
        if row["val_macro_auc_roc"] >= target:
            return elapsed
    return float("nan")

comparison_table = pd.DataFrame([{**{k: r[k] for k in ["name", "mode", "loss", "optimized", "best_val_auc_roc", "best_epoch", "epochs_completed", "total_train_seconds", "mean_epoch_seconds", "peak_memory_gb", "trainable_parameters", "total_parameters"]},
                                  "comparable_auc_roc_target": target_auc_roc,
                                  "time_to_comparable_auc_roc_seconds": time_to_target(r["history"], target_auc_roc)} for r in comparison])
comparison_table = comparison_table.rename(columns={"best_val_auc_roc": "validation_macro_auc-roc"})
comparison_table.to_csv(METRICS_DIR / "comparison_results.csv", index=False)
display(comparison_table.sort_values("validation_macro_auc-roc", ascending=False))


## Final model selection and thresholds tuning (on validation only)

In [ ]:
max_auc = comparison_table["validation_macro_auc-roc"].max()
eligible = comparison_table[comparison_table["validation_macro_auc-roc"] >= max_auc - QUALITY_TOLERANCE]
selected_name = eligible.sort_values("total_train_seconds").iloc[0]["name"]
selected = next(r for r in comparison if r["name"] == selected_name)
print(f"Selected: {selected_name}; validation auc-roc={selected['best_val_auc_roc']:.4f}")

def tune_thresholds(y_true, y_prob):
    grid = np.arange(0.05, 0.96, 0.01); thresholds = np.full(NUM_CLASSES, 0.5, dtype=np.float32)
    for c in range(NUM_CLASSES):
        if y_true[:, c].sum() and (1 - y_true[:, c]).sum():
            scores = [f1_score(y_true[:, c], y_prob[:, c] >= t, zero_division=0) for t in grid]
            thresholds[c] = grid[int(np.argmax(scores))]
    return thresholds

thresholds = tune_thresholds(selected["val_targets"], selected["val_probs"])
threshold_table = pd.DataFrame({"disease": DISEASES, "validation_threshold": thresholds})
display(threshold_table)
np.save(METRICS_DIR / "validation_thresholds.npy", thresholds)


## Steady-state classification and Grad-CAM timing

In [ ]:
def synchronize():
    if DEVICE.type == "cuda": torch.cuda.synchronize()

def classification_benchmark(model, loader, optimized):
    model.eval(); batches = iter(loader)
    def next_batch(iterator):
        try:
            return next(iterator), iterator
        except StopIteration:
            iterator = iter(loader)
            try:
                return next(iterator), iterator
            except StopIteration as error:
                raise RuntimeError("The benchmark loader is empty; no images are available for timing.") from error

    for _ in range(BENCHMARK["warmup_batches"]):
        (images, _, _), batches = next_batch(batches)
        images = images.to(DEVICE, non_blocking=PIN_MEMORY)
        if optimized and USE_CHANNELS_LAST: images = images.contiguous(memory_format=torch.channels_last)
        with torch.inference_mode(), torch.autocast(device_type=DEVICE.type, enabled=USE_AMP): _ = model(images)
    synchronize(); latencies, n = [], 0
    for _ in range(BENCHMARK["timing_batches"]):
        (images, _, _), batches = next_batch(batches)
        images = images.to(DEVICE, non_blocking=PIN_MEMORY)
        if optimized and USE_CHANNELS_LAST: images = images.contiguous(memory_format=torch.channels_last)
        synchronize(); start = time.perf_counter()
        with torch.inference_mode(), torch.autocast(device_type=DEVICE.type, enabled=USE_AMP): _ = model(images)
        synchronize(); latencies.append(time.perf_counter() - start); n += len(images)
    elapsed = sum(latencies)
    if n == 0 or elapsed <= 0:
        raise RuntimeError("No timed batches were recorded. Use a positive benchmark timing-batch setting and verify the validation loader.")
    return {"classification_ms_per_image": 1000 * elapsed / n, "classification_images_per_second": n / elapsed}

class GradCAM:
    def __init__(self, model):
        self.model, self.activations, self.gradients = model, None, None
        layer = model.features.denseblock4
        self.h1 = layer.register_forward_hook(lambda m, i, o: setattr(self, "activations", o))
        self.h2 = layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "gradients", go[0]))
    def close(self): self.h1.remove(); self.h2.remove()
    def generate(self, image, class_index):
        self.model.zero_grad(set_to_none=True); logits = self.model(image); logits[:, class_index].sum().backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)
        return (cam[0, 0] / (cam[0, 0].max() + 1e-8)).detach().cpu().numpy(), logits.detach()

def gradcam_benchmark(model, dataset, optimized):
    cam = GradCAM(model); times = []
    try:
        total_samples = BENCHMARK["gradcam_warmup"] + BENCHMARK["gradcam_samples"]
        for i in range(min(total_samples, len(dataset))):
            image, _, _ = dataset[i]; image = image.unsqueeze(0).to(DEVICE)
            if optimized and USE_CHANNELS_LAST: image = image.contiguous(memory_format=torch.channels_last)
            class_idx = 0
            synchronize(); start = time.perf_counter(); _, logits = cam.generate(image, class_idx); synchronize()
            if i >= BENCHMARK["gradcam_warmup"]: times.append(time.perf_counter() - start)
    finally: cam.close()
    if not times:
        raise RuntimeError("No Grad-CAM samples were timed; increase the benchmark sample settings.")
    return {"gradcam_ms_per_image": 1000 * np.mean(times)}

benchmark_rows = []
# Benchmark each explicit configuration, rather than `selected`, so the table
# remains a valid three-way end-to-end comparison even if selection picks full FT.
partial_ft_fp32 = next(r for r in comparison if r["name"] == "partial_ft_fp32")
partial_ft_optm = next(r for r in comparison if r["name"] == "partial_ft_optm")
for result in [full_reference, partial_ft_fp32, partial_ft_optm]:
    metrics = classification_benchmark(result["model"], val_loader, result["optimized"])
    metrics.update(gradcam_benchmark(result["model"], val_dataset, result["optimized"]))
    metrics["name"] = result["name"]; benchmark_rows.append(metrics)
benchmark_table = pd.DataFrame(benchmark_rows)
display(benchmark_table)


## One held-put test evaluation of the selected model (no test-based selection)

In [ ]:
test_targets, test_probs, test_indices = predict(selected["model"], test_loader)
test_predictions = (test_probs >= thresholds[None, :]).astype(int)
per_class_test = []
for c, disease in enumerate(DISEASES):
    has_both = np.unique(test_targets[:, c]).size == 2
    per_class_test.append({"disease": disease, "auc-roc": roc_auc_score(test_targets[:, c], test_probs[:, c]) if has_both else np.nan,
                           "auprc": average_precision_score(test_targets[:, c], test_probs[:, c]) if has_both else np.nan,
                           "f1_at_validation_threshold": f1_score(test_targets[:, c], test_predictions[:, c], zero_division=0),
                           "test_positives": int(test_targets[:, c].sum())})
per_class_test = pd.DataFrame(per_class_test)
test_summary = {"selected_model": selected_name, "test_macro_auc-roc": per_class_test["auc-roc"].mean(), "test_macro_auprc": per_class_test.auprc.mean(), "test_macro_f1": per_class_test.f1_at_validation_threshold.mean()}
print(test_summary)
display(per_class_test)
comparison_table.to_csv(METRICS_DIR / "comparison_results.csv", index=False)
benchmark_table.to_csv(BENCHMARK_DIR / "runtime_benchmark.csv", index=False)
per_class_test.to_csv(METRICS_DIR / "held_out_test_metrics.csv", index=False)


## Speed-quality frontier and final report table

In [ ]:
plot_df = comparison_table.copy()
fig, ax = plt.subplots(figsize=(9, 6))
styles = {
    "full_ft_fp32": {"color": "#1f77b4", "marker": "o"},
    "partial_ft_fp32": {"color": "#ff7f0e", "marker": "s"},
    "partial_ft_optm": {"color": "#2ca02c", "marker": "^"},
}
for _, row in plot_df.iterrows():
    style = styles[row["name"]]
    x, y = row["total_train_seconds"], row["validation_macro_auc-roc"]
    ax.scatter(x, y, s=125, color=style["color"], marker=style["marker"],
               label=row["name"], zorder=3)
ax.set_xlabel("Total training time (seconds)")
ax.set_ylabel("Validation macro auc-roc")
ax.set_title("Speed–quality frontier")
ax.grid(alpha=.3)
ax.legend(title="Configuration", loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=True)

fig.tight_layout(rect=(0, 0, 0.78, 1))
fig.savefig(FIGURE_DIR / "speed_quality_frontier.png", dpi=160, bbox_inches="tight")
plt.show()

report = comparison_table.merge(benchmark_table, on="name", how="left")
display(report.sort_values("validation_macro_auc-roc", ascending=False))
partial_row = report.loc[report["name"] == "partial_ft_optm"].iloc[0]
full_row = report.loc[report["name"] == "full_ft_fp32"].iloc[0]
if np.isfinite(partial_row["time_to_comparable_auc_roc_seconds"]):
    time_to_quality_statement = f"The partially optimized model reached the comparable-quality target in {partial_row['time_to_comparable_auc_roc_seconds']:.1f} seconds."
else:
    time_to_quality_statement = "The partially optimized model did not reach the comparable-quality target within the configured epoch budget."
report_text = "\n".join([
    f"# Experiment summary: {RUN_ID}", "",
    "## Objective",
    "Determine whether the partial DenseNet-121 fine-tuning model reaches comparable validation auc-roc sooner than full DenseNet-121 fine-tuning model, while reducing training, classification and Grad-CAM cost.", "",
    "## Comparable-quality definition",
    f"Target auc-roc = full FP32 best validation auc-roc ({full_row['validation_macro_auc-roc']:.4f}) - tolerance ({QUALITY_TOLERANCE:.4f}) = {target_auc_roc:.4f}.", "",
    "## Result", time_to_quality_statement,
    f"Full FT FP32 total training time: {full_row['total_train_seconds']:.1f} seconds. Partial FT optimized total training time: {partial_row['total_train_seconds']:.1f} seconds.", "",
    "## Reproducibility",
    "Configuration: `config/run_config.json`  ",
    "Exact split and seed: `manifests/split_manifest.csv`  ",
    "Class prevalence: `metrics/class_prevalence.csv`",
])
(REPORT_DIR / "experiment_summary.md").write_text(report_text, encoding="utf-8")
print("Saved run report:", REPORT_DIR / "experiment_summary.md")


## Qualitative Grad-CAM on a held-out test image

In [ ]:
sample_position = 0
original_sample = test_samples[int(test_indices[sample_position])]
image_tensor, target, _ = test_dataset[int(test_indices[sample_position])]
image_batch = image_tensor.unsqueeze(0).to(DEVICE)
if selected["optimized"] and USE_CHANNELS_LAST: image_batch = image_batch.contiguous(memory_format=torch.channels_last)
cam = GradCAM(selected["model"])
try:
    with torch.no_grad(): predicted_class = int(torch.sigmoid(selected["model"](image_batch))[0].argmax().item())
    heatmap, logits = cam.generate(image_batch, predicted_class)
finally:
    cam.close()

display_image = original_sample["image"]
if not isinstance(display_image, Image.Image): display_image = Image.fromarray(np.asarray(display_image))
display_image = display_image.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
probability = torch.sigmoid(logits)[0, predicted_class].item()
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(display_image); plt.axis("off"); plt.title("Held-out test X-ray")
plt.subplot(1, 2, 2); plt.imshow(display_image); plt.imshow(heatmap, cmap="jet", alpha=.45); plt.axis("off")
plt.title(f"Grad-CAM: {DISEASES[predicted_class]} ({probability:.2%})")
plt.tight_layout(); plt.show()
